# Data Quality Pipeline — Demo (Viva)

Run from project root so `src` imports resolve, or set `PYTHONPATH` to the repo root.

```bash
cd Automated-Data-Quality-Imputation-Pipeline
jupyter notebook notebooks/demo.ipynb
```

In [ ]:
import sys
from pathlib import Path
# Add project root to path (parent of notebooks/)
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from src.pipeline import PipelineConfig, run_pipeline

In [ ]:
rng = np.random.default_rng(0)
n = 100
df = pd.DataFrame({
    "feat1": rng.normal(0, 1, n),
    "feat2": rng.integers(0, 100, n).astype(float),
    "region": rng.choice(["N", "S", "E", "W"], n),
})
df.loc[rng.choice(n, 20, replace=False), "feat1"] = np.nan
df.loc[rng.choice(n, 15, replace=False), "region"] = np.nan
df = pd.concat([df, df.iloc[:3]], ignore_index=True)  # duplicates
df.head()

In [ ]:
cfg = PipelineConfig(
    knn_neighbors=5,
    contamination=0.05,
    scaler="standard",
    encoding="onehot",
    drop_duplicates=True,
)
result = run_pipeline(df, cfg)
cleaned = result["cleaned_df"]
print("Log:")
for line in result["log"]:
    print(" -", line)
cleaned.head()

In [ ]:
assert cleaned.isna().sum().sum() == 0
assert "is_outlier" in cleaned.columns
result["report_json"]["profile_before"]["n_rows"], len(cleaned)